# Is There A Replication Crisis In Finance?

**Authors**: Theis Ingerslev Jensen, Bryan Kelly, Lasse Heje Pedersen

**Published**: 2021-02-01

**URL**: [https://doi.org/10.3386/w28432](https://doi.org/10.3386/w28432)

**Abstract**: Several papers argue that financial economics faces a replication crisis because the majority of studies cannot be replicated or are the result of multiple testing of too many factors. We develop and estimate a Bayesian model of factor replication, which leads to different conclusions. The majority of asset pricing factors: (1) can be replicated, (2) can be clustered into 13 themes, the majority of which are significant parts of the tangency portfolio, (3) work out-of-sample in a new large data set covering 93 countries, and (4) have evidence that is strengthened (not weakened) by the large number of observed factors.

In [ ]:
!pip install yfinance

## Phase 1: Configuration

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Configuration
ticker = 'AAPL'
start_date = '2010-01-01'
end_date = '2023-01-01'
initial_capital = 10000

## Phase 2: Data Download and Feature Engineering

In [ ]:
# Download data
data = yf.download(ticker, start=start_date, end=end_date)

# Feature engineering
data['Return'] = data['Adj Close'].pct_change()
data['SMA_50'] = data['Adj Close'].rolling(window=50).mean()
data['SMA_200'] = data['Adj Close'].rolling(window=200).mean()

## Phase 3: Signal Generation and Portfolio Construction

In [ ]:
# Signal generation
data['Signal'] = 0
# Buy signal
data.loc[data['SMA_50'] > data['SMA_200'], 'Signal'] = 1
# Sell signal
data.loc[data['SMA_50'] < data['SMA_200'], 'Signal'] = -1

# Portfolio construction
data['Position'] = data['Signal'].shift(1)

## Phase 4: Vectorized Backtest

In [ ]:
# Calculate strategy returns
data['Strategy_Return'] = data['Position'] * data['Return']

# Calculate cumulative returns
data['Cumulative_Strategy_Return'] = (1 + data['Strategy_Return']).cumprod()
data['Cumulative_Market_Return'] = (1 + data['Return']).cumprod()

## Phase 5: Performance Metrics

In [ ]:
# Calculate performance metrics
sharpe_ratio = data['Strategy_Return'].mean() / data['Strategy_Return'].std() * np.sqrt(252)
sortino_ratio = data['Strategy_Return'].mean() / data[data['Strategy_Return'] < 0]['Strategy_Return'].std() * np.sqrt(252)
max_drawdown = (data['Cumulative_Strategy_Return'].cummax() - data['Cumulative_Strategy_Return']).max()
calmar_ratio = data['Strategy_Return'].mean() * 252 / max_drawdown

# Print metrics
print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
print(f'Sortino Ratio: {sortino_ratio:.2f}')
print(f'Max Drawdown: {max_drawdown:.2f}')
print(f'Calmar Ratio: {calmar_ratio:.2f}')

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(data['Cumulative_Strategy_Return'], label='Strategy')
plt.plot(data['Cumulative_Market_Return'], label='Market')
plt.title('Equity Curve')
plt.legend()
plt.show()

## Phase 6: Monitoring Stub

In [ ]:
# Monitoring stub
# This is a placeholder for real-time monitoring and alerting logic.
# In practice, you would implement code to monitor the strategy's performance and send alerts if necessary.

print('Monitoring stub: Implement real-time monitoring and alerting logic here.')